# Settings

In [16]:
from pathlib import Path

import vine_data as vd

H5_PATH = vd.DEFAULT_H5_PATH

# Load Labels

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Open read-only; keep the handle for later cells
f = vd.open_h5(H5_PATH)
wavelengths = vd.load_wavelengths(f)

print("\nroot attrs:")
for k, v in f.attrs.items():
    print(f"  {k}: {v}")
print(f"\nwavelengths: {wavelengths.shape[0]} bands, "
      f"{wavelengths.min():.1f}-{wavelengths.max():.1f} nm")

meta = vd.index_vines(f)
print(f"{len(meta)} vines indexed")
display(meta.head(5))


root attrs:
  class_names: ['Negative' 'Positive']
  label_names: ['red_blotch' 'leafroll3']
  label_year: 2026
  layout: cube is (lines, bands, samples) as stored in the ENVI BIL files
  sensor: Specim IQ
  sensor_docs: docs/Specim-IQ-Datasheet.pdf

wavelengths: 204 bands, 400.0-1000.0 nm
998 vines indexed


,path,variety,row,vine,ccovi_code,capture_date,red_blotch,leafroll3,label,cube_shape,dtype
0,Baco/R1/V1,Baco,R1,1,B01,2026-08-12,1,1,"[1, 1]","(151, 204, 157)",float32
1,Baco/R1/V2,Baco,R1,2,B02,2026-08-12,1,1,"[1, 1]","(158, 204, 151)",float32
2,Baco/R1/V3,Baco,R1,3,B03,2026-08-12,1,0,"[1, 0]","(141, 204, 150)",float32
3,Baco/R1/V4,Baco,R1,4,B04,2026-08-12,1,0,"[1, 0]","(158, 204, 153)",float32
4,Baco/R1/V5,Baco,R1,5,B05,2026-08-12,0,1,"[0, 1]","(140, 204, 171)",float32


In [29]:
def fmt(count, n):
    return f"{int(count)} ({100 * count / n:.1f}%)".replace(".0%", "%")

# Mutually exclusive status: RB only / LR only / both / healthy
rows_status = []
for variety, g in meta.groupby("variety", sort=False):
    n = len(g)
    rb_only = ((g.red_blotch == 1) & (g.leafroll3 == 0)).sum()
    lr_only = ((g.red_blotch == 0) & (g.leafroll3 == 1)).sum()
    both = ((g.red_blotch == 1) & (g.leafroll3 == 1)).sum()
    healthy = ((g.red_blotch == 0) & (g.leafroll3 == 0)).sum()
    rows_status.append({
        "Red Blotch": fmt(rb_only, n),
        "Leafroll": fmt(lr_only, n),
        "Both": fmt(both, n),
        "Healthy": fmt(healthy, n),
    })

status_distribution = pd.DataFrame(
    rows_status, index=list(meta.groupby("variety", sort=False).groups.keys())
)
status_distribution

,Red Blotch,Leafroll,Both,Healthy
Baco,82 (33.1%),54 (21.8%),53 (21.4%),59 (23.8%)
Cab Franc,240 (96%),0 (0%),1 (0.4%),9 (3.6%)
Chardonnay,38 (15.2%),72 (28.8%),50 (20%),90 (36%)
Vidal,157 (62.8%),13 (5.2%),12 (4.8%),68 (27.2%)
